# 03. Identity

**Tier:** Intermediate. **Why you'd use this:** You need to know whether two names point at one object or only at equal values. `==` asks about value. `is` asks about identity. Mixing them makes `None` checks wrong and invites CPython caching surprises.

**After this lesson you can:** choose `is` for singletons; use `==` for values; treat small-int caching as a CPython detail you never rely on.
About 25 minutes. **Uses:** [Names](../01-basics/01-names-lesson.ipynb)


## Quick review

1. Why does `[[0]*3]*2` share rows?
2. What does `is` compare that `==` does not?

<details><summary>Answers</summary>

Multiplication repeats one inner list reference. Object identity, not value equality.
</details>


## The idea

Every object has an identity. `is` is true when both sides are the same object. `==` is true when the values compare equal. Use `is` for `None` and other singletons. Use `==` for numbers, strings, and containers. CPython may cache small integers and some strings so `is` looks true by accident. That is an implementation detail, not a rule.

<details><summary>Explain more</summary>

`a is b` is like comparing `id(a)` and `id(b)`. Never write `x is 256` to mean equality. Write `x == 256`.
</details>


## See it work

Compare value and identity, including a `None` check.


In [1]:
def both_missing(a, b):
    # 1. None is a singleton: identity is the right test.
    return a is None and b is None


left = [1, 2]
right = [1, 2]
print(left == right)
print(left is right)
print(both_missing(None, None))


True
False
True


## Check yourself

Answer these before you run the cell.

1. Is `==` true for two equal lists that are not the same object?
2. Should you use `is` to compare integers you care about by value?


In [2]:
left = [1, 2]
right = [1, 2]
print(left == right)
print(left is right)
print((256 == 256), (257 == 257))


True
False
True True


<details><summary>Answers</summary>

Yes, `==` can be true while `is` is false. No. Use `==` for integer values.
</details>


## Why it works this way

Names are labels on objects. `is` asks whether two labels point at one object. `==` asks the type's equality method whether the values match. Lists with the same items are equal and still distinct objects after separate literals. Deep dive: [Names and objects](../../internals/01-objects/01-names-lesson.ipynb).

`None` is a singleton: the language has one `None` object. `is None` is the stable test. For ints and strings, CPython may reuse objects for small integers and some literals as a memory optimization. That can make `is` return true for equal values in one build and not another. Label that a CPython detail. Never teach it as a rule, and never write production checks that depend on it.

Cost of `is` is a pointer comparison. Cost of `==` depends on the type and may walk a whole container. The design reason to keep both operators is that identity and equality answer different questions. Confusing them is how `if x is 1` and accidental intern hits appear.

Evidence: equal lists with different ids, and a volatile small-int identity check tagged as CPython detail. String interning follows the same warning: two equal string literals may or may not be one object depending on how they were created. Write value comparisons when you mean value. Identity is a tool for singletons and for debugging aliasing. Equality is the tool for data. Keep those jobs separate and the CPython caches stop mattering. If a code review argues about `is` on an int, switch the check to `==` and move on.


In [3]:
left = [1, 2]
right = [1, 2]
print(left == right)
print(left is right)
print(id(left) == id(right))
print(None is None)


True
False
False
True


In [4]:
# CPython detail: small integers may be cached. Do not rely on this.
a = 256
b = 256
print(a is b)
x = 2578
y = 2578
print(x is y)


True
False


## When to use it

Use `is` for `None` and explicit singleton sentinels. Use `==` for values. Don't use `is` for numbers or strings you mean to compare by value. Don't rely on interned ints or strings. When a check must survive across Python implementations, write the comparison that matches the meaning.


## Common mistake

Writing `if total is 0` because it sometimes works for small ints. The check is about identity, not arithmetic equality.


In [5]:
total = 0
print(total is 0)
total = 0 + 1000 - 1000
print(total == 0)


True
True


<>:2: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
<>:2: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
/var/folders/7f/trms0d492gsf9j9ydv6kp1q00000gn/T/ipykernel_65515/1515589121.py:2: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  print(total is 0)


Compare numbers with `==`. Reserve `is` for singletons.


In [6]:
total = 0 + 1000 - 1000
print(total == 0)
print(total is None)


True
False


## Try it

Change `both_missing` to also accept a custom sentinel object you create once and pass in. Why must that sentinel be compared with `is`, not `==`?


## Recap

- `is` is identity. `==` is value.
- Use `is` for `None`. Use `==` for values.
- Small-int caching is a CPython detail, not an API.

**Next:** [practice](03-identity-practice.ipynb)
